# LSTM / GRU vs Mamba2 — Humanitarian Severity Forecasting

Compares four recurrent baselines against the existing Mamba2 model:
- **LSTM-Uni** — Unidirectional LSTM, 2 layers
- **LSTM-Bi** — Bidirectional LSTM, 2 layers
- **GRU-Uni** — Unidirectional GRU, 2 layers
- **GRU-Bi** — Bidirectional GRU, 2 layers

**Why bidirectional is valid here:** The 24-month input window is entirely in the past; labels are 3 future months.  
Bidirectional layers re-read the *historical* window in both directions — no label leakage.  
The question is empirical: does backward context over past months help predict future severity?

Identical data pipeline, loss function, and training config to `mamba_gap_prediction.ipynb`.  
Mamba2 reference: **MAE=0.1224, RMSE=0.2223, best_loss=0.0111** (stopped at epoch 54).

In [ ]:
import warnings, sys
warnings.filterwarnings('ignore')
sys.path.insert(0, 'analysis')

import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error
import matplotlib.pyplot as plt
import time

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

DATA   = Path('Data')
DEVICE = torch.device('mps' if torch.backends.mps.is_available() else 'cpu')
torch.manual_seed(42)
np.random.seed(42)

print(f'Device: {DEVICE}')
print(f'PyTorch: {torch.__version__}')

## 1. Data — identical pipeline to Mamba2 notebook

In [ ]:
X_monthly = np.load(DATA / 'monthly_severity_sequences.npy')   # (75, 65, 4)
ctx_arr   = np.load(DATA / 'context_features.npy')             # (75, 21)
countries  = pd.read_csv(DATA / 'sequence_countries.csv',  header=None)[0].tolist()
snapshots  = pd.read_csv(DATA / 'sequence_snapshots.csv',  header=None)[0].tolist()
enriched   = pd.read_csv(DATA / 'enriched_frame_2025.csv', index_col=0)
cerf_ufe   = pd.read_csv(DATA / 'Third-Party/Benchmarks/cerf_ufe.csv')
care_bts   = pd.read_csv(DATA / 'Third-Party/Benchmarks/care_bts.csv')

N, T, F_seq = X_monthly.shape
F_ctx = ctx_arr.shape[1]
print(f'Monthly sequences: {N} countries × {T} timesteps × {F_seq} features')
print(f'Context features:  {F_ctx}')
print(f'Date range:        {snapshots[0]} → {snapshots[-1]}')

In [ ]:
SEQ_LEN    = 24
PRED_STEPS = 3

seq_flat   = X_monthly[:, :48, :].reshape(-1, F_seq)
seq_scaler = StandardScaler().fit(seq_flat)
X_norm = seq_scaler.transform(X_monthly.reshape(-1, F_seq)).reshape(N, T, F_seq).astype(np.float32)

ctx_median = np.nanmedian(ctx_arr, axis=0)
for j in range(ctx_arr.shape[1]):
    ctx_arr[:, j] = np.where(np.isnan(ctx_arr[:, j]), ctx_median[j], ctx_arr[:, j])
ctx_scaler = StandardScaler().fit(ctx_arr)
ctx_norm   = ctx_scaler.transform(ctx_arr).astype(np.float32)

seqs_X, seqs_ctx, seqs_y_sev, seqs_meta = [], [], [], []
for i, iso3 in enumerate(countries):
    seq = X_norm[i]
    ctx = ctx_norm[i]
    for t in range(SEQ_LEN, T - PRED_STEPS):
        X_window  = seq[t - SEQ_LEN:t]
        y_sev_raw = X_monthly[i, t:t+PRED_STEPS, 0]
        seqs_X.append(X_window)
        seqs_ctx.append(ctx)
        seqs_y_sev.append(float(np.mean(y_sev_raw)))
        seqs_meta.append({'iso3': iso3, 'end_snapshot': snapshots[t],
                          'target_snapshot': snapshots[min(t+PRED_STEPS-1, T-1)]})

X_arr   = np.stack(seqs_X).astype(np.float32)
ctx_mat = np.stack(seqs_ctx).astype(np.float32)
y_arr   = np.array(seqs_y_sev, dtype=np.float32)
meta_df = pd.DataFrame(seqs_meta)

train_mask = meta_df['end_snapshot'] < '2025-01'
train_idx  = meta_df[train_mask].index.tolist()
test_idx   = meta_df[~train_mask].index.tolist()

neglect_map = {}
for iso3 in countries:
    if iso3 in enriched.index:
        cov = enriched.loc[iso3, 'coverage']
        gap = enriched.loc[iso3, 'gap_score_cerf_profile']
        neglect_map[iso3] = {
            'neglected': float(not pd.isna(cov) and cov < 0.4),
            'gap_score': float(gap) if not pd.isna(gap) else 0.5
        }

y_neglect = np.array([neglect_map.get(m['iso3'], {}).get('neglected', 0.5)
                      for m in seqs_meta], dtype=np.float32)
y_gap     = np.array([neglect_map.get(m['iso3'], {}).get('gap_score', 0.5)
                      for m in seqs_meta], dtype=np.float32)

print(f'Train: {len(train_idx)},  Test: {len(test_idx)}')
print(f'Target range: {y_arr.min():.2f} – {y_arr.max():.2f}')

In [ ]:
class MultiTargetDataset(Dataset):
    def __init__(self, X, ctx, y_sev, y_gap, y_neg):
        self.X     = torch.tensor(X,     dtype=torch.float32)
        self.ctx   = torch.tensor(ctx,   dtype=torch.float32)
        self.y_sev = torch.tensor(y_sev, dtype=torch.float32)
        self.y_gap = torch.tensor(y_gap, dtype=torch.float32)
        self.y_neg = torch.tensor(y_neg, dtype=torch.float32)
    def __len__(self): return len(self.X)
    def __getitem__(self, i):
        return self.X[i], self.ctx[i], self.y_sev[i], self.y_gap[i], self.y_neg[i]


def make_loaders(batch_train=64, batch_test=128):
    train_ds = MultiTargetDataset(
        X_arr[train_idx], ctx_mat[train_idx],
        y_arr[train_idx], y_gap[train_idx], y_neglect[train_idx])
    test_ds = MultiTargetDataset(
        X_arr[test_idx], ctx_mat[test_idx],
        y_arr[test_idx], y_gap[test_idx], y_neglect[test_idx])
    return (DataLoader(train_ds, batch_size=batch_train, shuffle=True,  drop_last=False),
            DataLoader(test_ds,  batch_size=batch_test,  shuffle=False))

train_loader, test_loader = make_loaders()

n_pos = float(y_neglect[train_idx].sum())
n_neg = float(len(train_idx)) - n_pos
pos_weight = torch.tensor([n_neg / max(n_pos, 1.0)], dtype=torch.float32, device=DEVICE)
print(f'pos_weight for neglect BCE: {pos_weight.item():.3f}')

## 2. Model Definitions

All four models share identical structure:
- **Context conditioning:** ctx projected to `d_model`, added to the input embedding at every timestep.
  This mirrors Mamba2's `ctx_gate` which biases the dt (selectivity) via additive modulation.
- **Pooling:** `[last_timestep ∥ mean_pool]` — same as Mamba2.
- **Heads:** same two-layer MLP for severity, gap, neglect.

**Bidirectional note:** For Bi models the RNN output is `2×hidden`, so pool dim is `4×hidden`.
We use `hidden=48` for Uni (pool→96) and `hidden=48` for Bi (pool→192) so Bi has slightly more params in the heads — intentional, since bidir naturally has 2× recurrent capacity.

In [ ]:
class RNNForecaster(nn.Module):
    """
    Shared base for LSTM/GRU × Uni/Bi.

    Context conditioning: static features are projected to d_model and added
    to the input embedding at each timestep before feeding the RNN.  This gives
    the same "ctx modulates sequence processing" behaviour as Mamba2's dt gate.

    Pooling: concat(last_hidden, mean_over_time) → prediction heads.
    """
    def __init__(self, F_seq, F_ctx, rnn_type='lstm', bidirectional=False,
                 hidden=48, n_layers=2, dropout=0.15):
        super().__init__()
        self.bidirectional = bidirectional
        self.hidden = hidden
        D = 2 if bidirectional else 1       # direction multiplier
        rnn_out = hidden * D                # RNN output feature dim
        pool_dim = rnn_out * 2              # last + mean

        self.input_proj = nn.Linear(F_seq, hidden)
        self.ctx_proj   = nn.Sequential(
            nn.Linear(F_ctx, hidden), nn.LayerNorm(hidden), nn.GELU())

        rnn_cls = nn.LSTM if rnn_type == 'lstm' else nn.GRU
        self.rnn = rnn_cls(
            input_size=hidden, hidden_size=hidden,
            num_layers=n_layers, batch_first=True,
            dropout=dropout if n_layers > 1 else 0.0,
            bidirectional=bidirectional)

        self.norm    = nn.LayerNorm(rnn_out)
        self.dropout = nn.Dropout(dropout)

        def head(in_dim):
            return nn.Sequential(
                nn.Linear(in_dim, 32), nn.GELU(), nn.Linear(32, 1))

        self.sev_head     = head(pool_dim)
        self.gap_head     = head(pool_dim)
        self.neglect_head = head(pool_dim)

    def forward(self, x, ctx):
        # x:   (B, L, F_seq)
        # ctx: (B, F_ctx)
        ctx_emb = self.ctx_proj(ctx)                        # (B, hidden)
        h = self.input_proj(x)                              # (B, L, hidden)
        h = h + ctx_emb.unsqueeze(1)                        # broadcast context
        h = self.dropout(h)

        out, _ = self.rnn(h)                                # (B, L, rnn_out)
        out    = self.norm(out)

        h_last = out[:, -1, :]                              # last step
        h_mean = out.mean(dim=1)                            # temporal mean
        pool   = torch.cat([h_last, h_mean], dim=-1)        # (B, 2*rnn_out)

        return (
            self.sev_head(pool).squeeze(-1),
            self.gap_head(pool).squeeze(-1),
            self.neglect_head(pool).squeeze(-1),
        )


def count_params(m):
    return sum(p.numel() for p in m.parameters())


configs = {
    'LSTM-Uni': dict(rnn_type='lstm', bidirectional=False),
    'LSTM-Bi':  dict(rnn_type='lstm', bidirectional=True),
    'GRU-Uni':  dict(rnn_type='gru',  bidirectional=False),
    'GRU-Bi':   dict(rnn_type='gru',  bidirectional=True),
}

print('Model parameter counts:')
print(f'  Mamba2 (reference):  110,739')
for name, cfg in configs.items():
    m = RNNForecaster(F_seq=F_seq, F_ctx=F_ctx, hidden=48, n_layers=2, dropout=0.15, **cfg)
    print(f'  {name:12s}: {count_params(m):>8,}')

## 3. Training — identical hyperparameters to Mamba2

| Hyperparameter | Value |
|---|---|
| Optimizer | AdamW, lr=8e-4, wd=3e-3 |
| Scheduler | OneCycleLR, pct_start=0.15 |
| Loss | 0.5×Huber(sev) + 0.3×Huber(gap) + 0.2×BCE(neglect) |
| Early stop patience | 30 epochs |
| Max epochs | 150 |

In [ ]:
EPOCHS   = 150
LR       = 8e-4
PATIENCE = 30
W_SEV, W_GAP, W_NEG = 0.5, 0.3, 0.2


def train_model(model, name):
    """Train a model with the same schedule used for Mamba2."""
    model = model.to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=3e-3)
    scheduler = torch.optim.lr_scheduler.OneCycleLR(
        optimizer, max_lr=LR, epochs=EPOCHS,
        steps_per_epoch=len(train_loader), pct_start=0.15)

    best_loss, best_state, patience_ctr = float('inf'), None, 0
    train_losses, test_losses, test_maes = [], [], []
    t0 = time.time()

    for epoch in range(EPOCHS):
        model.train()
        ep_loss = 0.0
        for X_b, ctx_b, y_sev_b, y_gap_b, y_neg_b in train_loader:
            X_b, ctx_b = X_b.to(DEVICE), ctx_b.to(DEVICE)
            y_sev_b, y_gap_b, y_neg_b = y_sev_b.to(DEVICE), y_gap_b.to(DEVICE), y_neg_b.to(DEVICE)

            sev_pred, gap_pred, neg_logit = model(X_b, ctx_b)
            loss = (W_SEV * F.huber_loss(sev_pred, y_sev_b, delta=0.5) +
                    W_GAP * F.huber_loss(gap_pred, y_gap_b, delta=0.5) +
                    W_NEG * F.binary_cross_entropy_with_logits(neg_logit, y_neg_b, pos_weight=pos_weight))

            optimizer.zero_grad(); loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 0.5)
            optimizer.step(); scheduler.step()
            ep_loss += loss.item()

        model.eval()
        t_loss, all_pred, all_true = 0.0, [], []
        with torch.no_grad():
            for X_b, ctx_b, y_sev_b, y_gap_b, y_neg_b in test_loader:
                X_b, ctx_b = X_b.to(DEVICE), ctx_b.to(DEVICE)
                y_sev_b, y_gap_b, y_neg_b = y_sev_b.to(DEVICE), y_gap_b.to(DEVICE), y_neg_b.to(DEVICE)
                sev_pred, gap_pred, neg_logit = model(X_b, ctx_b)
                t_loss += (W_SEV * F.huber_loss(sev_pred, y_sev_b, delta=0.5) +
                           W_GAP * F.huber_loss(gap_pred, y_gap_b, delta=0.5) +
                           W_NEG * F.binary_cross_entropy_with_logits(neg_logit, y_neg_b, pos_weight=pos_weight)).item()
                all_pred.extend(sev_pred.cpu().numpy())
                all_true.extend(y_sev_b.cpu().numpy())

        tl  = t_loss / len(test_loader)
        mae = mean_absolute_error(all_true, all_pred)
        train_losses.append(ep_loss / len(train_loader))
        test_losses.append(tl)
        test_maes.append(mae)

        if tl < best_loss:
            best_loss, best_state, patience_ctr = tl, {k: v.clone() for k, v in model.state_dict().items()}, 0
        else:
            patience_ctr += 1
            if patience_ctr >= PATIENCE:
                print(f'  [{name}] Early stop at epoch {epoch}')
                break

        if epoch % 20 == 0:
            print(f'  [{name}] Epoch {epoch:3d} | train={train_losses[-1]:.4f} | test={tl:.4f} | MAE={mae:.4f}')

    model.load_state_dict(best_state)
    elapsed = time.time() - t0
    print(f'  [{name}] Done — best_loss={best_loss:.4f}  ({elapsed:.0f}s)')
    return model, best_loss, train_losses, test_losses, test_maes

In [ ]:
results = {}

for name, cfg in configs.items():
    print(f'\n=== Training {name} ===')
    model = RNNForecaster(F_seq=F_seq, F_ctx=F_ctx, hidden=48, n_layers=2, dropout=0.15, **cfg)
    trained, best_loss, tr_losses, te_losses, te_maes = train_model(model, name)
    results[name] = {
        'model': trained,
        'best_loss': best_loss,
        'train_losses': tr_losses,
        'test_losses': te_losses,
        'test_maes': te_maes,
        'params': count_params(trained),
    }

## 4. Evaluation — metrics + CERF UFE overlap

In [ ]:
def evaluate(model, name):
    model.eval()
    all_pred, all_true, all_gap, all_neg = [], [], [], []
    with torch.no_grad():
        for X_b, ctx_b, y_sev_b, y_gap_b, y_neg_b in test_loader:
            sev_pred, gap_pred, neg_logit = model(X_b.to(DEVICE), ctx_b.to(DEVICE))
            all_pred.extend(sev_pred.cpu().numpy())
            all_true.extend(y_sev_b.numpy())
            all_gap.extend(gap_pred.cpu().numpy())
            all_neg.extend(torch.sigmoid(neg_logit).cpu().numpy())

    pred = np.array(all_pred)
    true = np.array(all_true)
    mae  = mean_absolute_error(true, pred)
    rmse = np.sqrt(mean_squared_error(true, pred))
    naive_mae = mean_absolute_error(true, np.full_like(true, true.mean()))
    return {
        'MAE':       mae,
        'RMSE':      rmse,
        'Naive_MAE': naive_mae,
        'MAE_vs_naive': naive_mae / mae,
        'pred': pred,
        'gap':  np.array(all_gap),
        'neg':  np.array(all_neg),
    }


eval_results = {}
for name in results:
    eval_results[name] = evaluate(results[name]['model'], name)

# Include Mamba2 reference values
mamba_ref = {'MAE': 0.1224, 'RMSE': 0.2223, 'Naive_MAE': 0.8060,
             'MAE_vs_naive': 0.8060/0.1224, 'best_loss': 0.0111, 'params': 110739}

print('\nMetric comparison (lower MAE/RMSE = better):')
print(f'{"Model":<12} {"Params":>8}  {"MAE":>7}  {"RMSE":>7}  {"vs Naive":>9}  {"BestLoss":>9}')
print('-' * 60)
print(f'{"Mamba2":12} {110739:>8,}  {0.1224:>7.4f}  {0.2223:>7.4f}  {6.59:>8.2f}×  {0.0111:>9.4f}')
for name in results:
    ev = eval_results[name]
    bl = results[name]['best_loss']
    ps = results[name]['params']
    print(f'{name:<12} {ps:>8,}  {ev["MAE"]:>7.4f}  {ev["RMSE"]:>7.4f}  '
          f'{ev["MAE_vs_naive"]:>8.2f}×  {bl:>9.4f}')

In [ ]:
def safe_norm(s):
    mn, mx = s.min(), s.max()
    return (s - mn) / (mx - mn) if mx > mn else pd.Series(0.0, index=s.index)


def build_ranking(model, name):
    """Run country-level inference and compute composite ranking, same as Mamba2 notebook."""
    model.eval()
    rows = []
    for i, iso3 in enumerate(countries):
        seq_t = torch.tensor(X_norm[i, -SEQ_LEN:][None], dtype=torch.float32).to(DEVICE)
        ctx_t = torch.tensor(ctx_norm[i][None],           dtype=torch.float32).to(DEVICE)
        with torch.no_grad():
            sev_pred, gap_pred, neg_logit = model(seq_t, ctx_t)
        row = {
            'iso3':                  iso3,
            'current_severity':      float(X_monthly[i, -1, 0]),
            'predicted_severity_3m': float(sev_pred.cpu()),
            'neglect_prob':          float(torch.sigmoid(neg_logit).cpu()),
        }
        if iso3 in enriched.index:
            for col in ['gap_score_cerf_profile', 'severity_baseline_24m']:
                row[col] = enriched.loc[iso3, col] if col in enriched.columns else np.nan
        row['severity_worsening'] = max(0.0, row['predicted_severity_3m'] - row['current_severity'])
        rows.append(row)

    df = pd.DataFrame(rows)
    df['sev_worsen_norm'] = safe_norm(df['severity_worsening'].fillna(0))
    df['gap_norm']        = safe_norm(df['gap_score_cerf_profile'].fillna(0))
    df['neglect_norm']    = safe_norm(df['neglect_prob'].fillna(0))
    df['composite_score'] = (0.40 * df['gap_norm'] +
                             0.35 * df['sev_worsen_norm'] +
                             0.25 * df['neglect_norm'])

    crisis_mask = df['severity_baseline_24m'].fillna(0) >= 3
    ranking = df[crisis_mask].sort_values('composite_score', ascending=False).reset_index(drop=True)
    ranking['rank'] = ranking.index + 1
    return ranking


ufe_set = set(cerf_ufe['iso3'])
bts_set = set(care_bts['iso3'])

print('\nCERF UFE overlap (higher = better):')
print(f'{"Model":<12}  {@5}  {@10}  {@15}  | CARE@10')
print('-' * 58)

# Mamba2 reference
mamba_ranking = pd.read_csv(DATA / 'gap_ranking_mamba2_v2.csv')
for k, label in [(5,'@5 '), (10,'@10'), (15,'@15')]:
    overlap = len(set(mamba_ranking['iso3'].head(k)) & ufe_set)
mamba_overlaps = {
    k: len(set(mamba_ranking['iso3'].head(k)) & ufe_set) for k in [5,10,15]}
mamba_bts = len(set(mamba_ranking['iso3'].head(10)) & bts_set)
print(f'{"Mamba2":<12}  {mamba_overlaps[5]}/5   {mamba_overlaps[10]}/10   {mamba_overlaps[15]}/15  | {mamba_bts}/10')

overlap_results = {}
for name in results:
    ranking = build_ranking(results[name]['model'], name)
    results[name]['ranking'] = ranking
    ov = {k: len(set(ranking['iso3'].head(k)) & ufe_set) for k in [5, 10, 15]}
    bts = len(set(ranking['iso3'].head(10)) & bts_set)
    overlap_results[name] = ov
    print(f'{name:<12}  {ov[5]}/5   {ov[10]}/10   {ov[15]}/15  | {bts}/10')

## 5. Training Curves

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
axes = axes.flatten()
colors = ['#2196F3', '#FF9800', '#4CAF50', '#E91E63']

for ax, (name, res), color in zip(axes, results.items(), colors):
    ax.plot(res['train_losses'], label='Train', lw=2, color=color, alpha=0.6)
    ax.plot(res['test_losses'],  label='Test',  lw=2, color=color)
    ax2 = ax.twinx()
    ax2.plot(res['test_maes'], label='MAE', lw=1.5, color='gray', ls='--')
    ax2.axhline(0.1224, color='red', ls=':', lw=1, label='Mamba2 MAE')
    ax2.set_ylabel('Severity MAE', fontsize=9)
    ax.set_title(f'{name}  (params={res["params"]:,})', fontweight='bold')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Loss')
    lines1, labels1 = ax.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax.legend(lines1 + lines2, labels1 + labels2, fontsize=8, loc='upper right')
    best_mae = min(res['test_maes'])
    ax.set_title(f'{name}  (best MAE={best_mae:.4f})', fontweight='bold')

plt.suptitle('LSTM / GRU training curves — red dashed = Mamba2 MAE target (0.1224)', y=1.01)
plt.tight_layout()
plt.savefig('lstm_gru_training_curves.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Final Comparison Table

In [ ]:
rows = []

# Mamba2 reference
rows.append({
    'Model':        'Mamba2 (ref)',
    'Params':       110_739,
    'MAE':          0.1224,
    'RMSE':         0.2223,
    'vs_Naive':     f'{6.59:.2f}×',
    'BestLoss':     0.0111,
    'StopEpoch':    54,
    'UFE@5':        mamba_overlaps[5],
    'UFE@10':       mamba_overlaps[10],
    'UFE@15':       mamba_overlaps[15],
    'CARE@10':      mamba_bts,
})

for name in results:
    ev  = eval_results[name]
    res = results[name]
    ov  = overlap_results[name]
    bts = len(set(res['ranking']['iso3'].head(10)) & bts_set)
    stop_epoch = len(res['test_losses'])
    rows.append({
        'Model':     name,
        'Params':    res['params'],
        'MAE':       ev['MAE'],
        'RMSE':      ev['RMSE'],
        'vs_Naive':  f"{ev['MAE_vs_naive']:.2f}×",
        'BestLoss':  res['best_loss'],
        'StopEpoch': stop_epoch,
        'UFE@5':     ov[5],
        'UFE@10':    ov[10],
        'UFE@15':    ov[15],
        'CARE@10':   bts,
    })

compare_df = pd.DataFrame(rows)
compare_df = compare_df.set_index('Model')
print('\n=== FULL COMPARISON TABLE ===')
print(compare_df.to_string())

# Best model by MAE
mae_vals = {r['Model']: r['MAE'] for r in rows}
best_name = min(mae_vals, key=mae_vals.get)
print(f'\nBest MAE: {best_name} ({mae_vals[best_name]:.4f})')

compare_df.to_csv(DATA / 'model_comparison.csv')
print('Saved: Data/model_comparison.csv')

In [ ]:
# Pred vs True scatter for all 4 models
fig, axes = plt.subplots(1, 4, figsize=(18, 4))
colors = ['#2196F3', '#FF9800', '#4CAF50', '#E91E63']

for ax, (name, res), color in zip(axes, results.items(), colors):
    ev   = eval_results[name]
    true = np.array([y for batch in test_loader for y in batch[2].numpy()])
    pred = ev['pred']
    ax.scatter(true, pred, alpha=0.2, s=10, color=color, edgecolors='none')
    lims = [min(true.min(), pred.min()) - 0.1, max(true.max(), pred.max()) + 0.1]
    ax.plot(lims, lims, 'k--', lw=1)
    ax.set_xlim(lims); ax.set_ylim(lims)
    ax.set_xlabel('True severity'); ax.set_ylabel('Predicted')
    ax.set_title(f'{name}\nMAE={ev["MAE"]:.4f}  RMSE={ev["RMSE"]:.4f}', fontweight='bold')
    ax.set_aspect('equal')

plt.suptitle('Predicted vs True Severity (test set, 2025-01 onward)', y=1.01)
plt.tight_layout()
plt.savefig('lstm_gru_scatter.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Save Models

In [ ]:
for name, res in results.items():
    fname = name.lower().replace('-', '_') + '_model.pt'
    torch.save(res['model'].state_dict(), fname)
    print(f'Saved: {fname}')